# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tabassumrafiq/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 2. My model under an honest split — before/after

The Week-5 model was originally evaluated using a client-grouped split.

For this validation audit, I compare that result with a random row split using
the same target, features, model family, and evaluation metric.

The random split provides a less conservative comparison because rows from the
same client can appear in both training and test data.

The client-grouped split is more appropriate for testing whether the model can
generalize to clients that were not present during training.

The comparison is therefore treated as a validation finding rather than proof
that one model is universally better.

In [3]:
import pandas as pd

url = "https://raw.githubusercontent.com/tabassumrafiq/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

data = pd.read_csv(url)

print("Dataset shape:", data.shape)
display(data.head())

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [6]:
import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

RANDOM_STATE = 42

data = pd.read_csv(url)

print("Dataset shape:", data.shape)

Dataset shape: (30000, 44)


In [7]:
data["is_declining_label"] = (
    data["trend_direction"] == "down"
).astype(int)

print("Target distribution:")
display(data["is_declining_label"].value_counts())

print("\nOverall base rate:")
print(round(data["is_declining_label"].mean(), 4))

Target distribution:


,count
is_declining_label,
1,16262
0,13738



Overall base rate:
0.5421


In [8]:
features = [
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "word_count"
]

X = data[features].copy()
y = data["is_declining_label"].copy()
groups = data["client_id"].copy()

print("Features:")
print(features)

print("\nRows:", len(X))
print("Base rate:", round(y.mean(), 4))

Features:
['avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'word_count']

Rows: 30000
Base rate: 0.5421


In [9]:
def create_rf_model():
    return Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=6,
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ])

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### BEFORE — Random row split

The random split allows rows from the same client to appear in both the
training and test sets.

This is useful as a comparison point, but it may produce an optimistic
estimate when observations from the same client share characteristics.

In [10]:
X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y
    )
)

random_model = create_rf_model()

random_model.fit(
    X_train_random,
    y_train_random
)

random_prob = random_model.predict_proba(
    X_test_random
)[:, 1]

random_pred = (
    random_prob >= 0.5
).astype(int)

random_precision = precision_score(
    y_test_random,
    random_pred,
    zero_division=0
)

random_recall = recall_score(
    y_test_random,
    random_pred,
    zero_division=0
)

random_f1 = f1_score(
    y_test_random,
    random_pred,
    zero_division=0
)

random_accuracy = accuracy_score(
    y_test_random,
    random_pred
)

print("BEFORE — Random row split")
print("Accuracy:", round(random_accuracy, 4))
print("Precision:", round(random_precision, 4))
print("Recall:", round(random_recall, 4))
print("F1:", round(random_f1, 4))
print("Base rate:", round(y_test_random.mean(), 4))

BEFORE — Random row split
Accuracy: 0.6352
Precision: 0.6121
Recall: 0.8924
F1: 0.7261
Base rate: 0.542


### AFTER — Client-grouped split

The grouped split keeps all observations from each client entirely within
either the training or test set.

This prevents the model from being evaluated on a client whose other rows
were already seen during training.

For this task, this is the more conservative validation design.

In [11]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_grouped = X.iloc[train_idx].copy()
X_test_grouped = X.iloc[test_idx].copy()

y_train_grouped = y.iloc[train_idx].copy()
y_test_grouped = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Train rows:", len(X_train_grouped))
print("Test rows:", len(X_test_grouped))

print("Train clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

client_overlap = set(groups_train).intersection(
    set(groups_test)
)

print("Client overlap:", len(client_overlap))

assert len(client_overlap) == 0

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0


In [12]:
grouped_model = create_rf_model()

grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_prob = grouped_model.predict_proba(
    X_test_grouped
)[:, 1]

grouped_pred = (
    grouped_prob >= 0.5
).astype(int)

grouped_precision = precision_score(
    y_test_grouped,
    grouped_pred,
    zero_division=0
)

grouped_recall = recall_score(
    y_test_grouped,
    grouped_pred,
    zero_division=0
)

grouped_f1 = f1_score(
    y_test_grouped,
    grouped_pred,
    zero_division=0
)

grouped_accuracy = accuracy_score(
    y_test_grouped,
    grouped_pred
)

print("AFTER — Client-grouped split")
print("Accuracy:", round(grouped_accuracy, 4))
print("Precision:", round(grouped_precision, 4))
print("Recall:", round(grouped_recall, 4))
print("F1:", round(grouped_f1, 4))
print("Base rate:", round(y_test_grouped.mean(), 4))

AFTER — Client-grouped split
Accuracy: 0.5398
Precision: 0.5283
Recall: 0.927
F1: 0.673
Base rate: 0.511


In [13]:
validation_comparison = pd.DataFrame([
    {
        "validation": "Random row split",
        "accuracy": random_accuracy,
        "precision": random_precision,
        "recall": random_recall,
        "f1": random_f1,
        "base_rate": y_test_random.mean()
    },
    {
        "validation": "Client-grouped split",
        "accuracy": grouped_accuracy,
        "precision": grouped_precision,
        "recall": grouped_recall,
        "f1": grouped_f1,
        "base_rate": y_test_grouped.mean()
    }
])

display(validation_comparison.round(4))

,validation,accuracy,precision,recall,f1,base_rate
0,Random row split,0.6352,0.6121,0.8924,0.7261,0.542
1,Client-grouped split,0.5398,0.5283,0.9270,0.6730,0.511


In [14]:
def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    k = min(k, len(scores))

    order = np.argsort(-scores)[:k]

    return labels[order].mean()

In [15]:
K_values = [10, 20, 50]

ranking_rows = []

for k in K_values:

    ranking_rows.append({
        "validation": "Random row split",
        "K": k,
        "precision_at_k": precision_at_k(
            random_prob,
            y_test_random,
            k
        ),
        "base_rate": y_test_random.mean()
    })

    ranking_rows.append({
        "validation": "Client-grouped split",
        "K": k,
        "precision_at_k": precision_at_k(
            grouped_prob,
            y_test_grouped,
            k
        ),
        "base_rate": y_test_grouped.mean()
    })

ranking_comparison = pd.DataFrame(ranking_rows)

display(
    ranking_comparison.round(4)
)

,validation,K,precision_at_k,base_rate
0,Random row split,10,0.80,0.542
1,Client-grouped split,10,0.50,0.511
2,Random row split,20,0.80,0.542
3,Client-grouped split,20,0.55,0.511
4,Random row split,50,0.84,0.542
5,Client-grouped split,50,0.56,0.511


### Interpretation

The random row split and client-grouped split provide two views of measured
model performance.

The random split can be optimistic because observations from the same client
may appear in both training and test data.

The client-grouped result is the more conservative estimate for generalization
to unseen clients.

Any gap between the two results is treated as a validation finding. It does
not by itself prove that the model is memorizing clients, but it indicates
that the choice of validation design affects the measured performance.

Precision@K is interpreted relative to the test-set base rate.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

I audited the final ML-08 feature set using three leakage checks.

First, I checked for label-derived fields. The target is derived from
`trend_direction`, which itself comes from `trend_pct`, so these fields must
not be model inputs.

Second, I checked for future or overlapping windows. Fields such as
`impressions_last_30d`, `clicks_last_30d`, and `sessions_last_30d` are excluded
because they can overlap with the period used to define the outcome.

Third, I checked that client and content identifiers are not predictive
features. They are used only for grouping and identification.

The final feature set contains only:
`avg_position`, `ctr`, `engagement_rate`, `scroll_rate`, and `word_count`.

In [16]:
label_columns = [
    "trend_direction",
    "trend_pct"
]

future_columns = [
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d"
]

identifier_columns = [
    "client_id",
    "content_id"
]

final_features = features.copy()

print("Final features:")
print(final_features)

print("\nLabel-derived fields excluded:")
print(label_columns)

print("\nFuture-window fields excluded:")
print(future_columns)

print("\nIdentifier fields excluded:")
print(identifier_columns)

Final features:
['avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'word_count']

Label-derived fields excluded:
['trend_direction', 'trend_pct']

Future-window fields excluded:
['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d']

Identifier fields excluded:
['client_id', 'content_id']


In [17]:
leakage_terms = [
    "trend",
    "last_30",
    "last30",
    "future",
    "outcome",
    "label",
    "target",
    "needs_refresh",
    "product_flag"
]

possible_leaks = [
    col
    for col in final_features
    if any(
        term in col.lower()
        for term in leakage_terms
    )
]

print("Possible leakage columns:")
print(possible_leaks)

assert len(possible_leaks) == 0

print("\nLeakage check passed.")

Possible leakage columns:
[]

Leakage check passed.


In [18]:
client_overlap = set(
    groups_train
).intersection(
    set(groups_test)
)

print("Client overlap:", len(client_overlap))

assert len(client_overlap) == 0

print("Grouped client leakage check passed.")

Client overlap: 0
Grouped client leakage check passed.


In [19]:
print("Overall base rate:", round(y.mean(), 4))
print("Random-split test base rate:", round(y_test_random.mean(), 4))
print("Grouped-split test base rate:", round(y_test_grouped.mean(), 4))

Overall base rate: 0.5421
Random-split test base rate: 0.542
Grouped-split test base rate: 0.511


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Original bold claim

The Random Forest model accurately predicts which content will decline and can
identify the pages that should be refreshed.

### Safer claim

On the evaluated dataset, the Random Forest model measured directional
out-of-sample performance for ranking content by the observed decline label.

Under client-grouped validation, the model achieved the measured
precision@K values shown in the validation table, with performance interpreted
relative to the test-set base rate.

These results support using the model as decision-support for prioritizing
content for human review.

They do not establish that the model will always predict future decline
correctly, and they do not show that refreshing a flagged page will cause an
improvement in performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Attack-your-own-model checklist

- [x] The label source was identified.
- [x] `trend_direction` was excluded from the features.
- [x] `trend_pct` was excluded from the features.
- [x] Future-window fields were excluded.
- [x] Client IDs were not used as predictive features.
- [x] Content IDs were not used as predictive features.
- [x] Client-grouped validation was performed.
- [x] Client overlap was checked and was zero.
- [x] The base rate was reported.
- [x] Precision@K was compared across validation designs.
- [x] False positives and false negatives were inspected in ML-08.
- [x] Feature importance was treated as directional, not causal.
- [x] Claims were rewritten using observed, measured, directional, and
      decision-support language.
- [ ] Runtime → Run all completed without errors.
- [ ] Final notebook was reviewed.
- [ ] Notebook was committed to `work/notebooks/w06_validation_audit.ipynb`.